Jupyter Notebook to solve the steady Stokes equation on a sphere as described in:
*  

Importing necessary libraries

In [1]:
import sys
##### path to the generate_surface_meshes.py  file in your system #######
sys.path.insert(0, "/Users/sami/Documents/Github/cosmos/sandbox/")
import time
# caution: path[0] is reserved for script path (or '' in REPL)
from netgen.occ import *
from netgen import meshing
from netgen.csg import *
from ngsolve import *
from ngsolve.webgui import Draw

from generate_surface_meshes import *

Defining auxuliary function for symbolic computation of gradient

In [2]:

def gradient(f,P):

    m, _ = P.dims
    l = len(f.dims)

    if l == 0:

        if m == 2:
            gradient = P*CoefficientFunction((f.Diff(x), f.Diff(y)))
        elif m == 3:
            gradient = P*CoefficientFunction((f.Diff(x), f.Diff(y), f.Diff(z))) 
            
    elif l == 1:

        if m == 2:
            gradient = P*CoefficientFunction((f[0].Diff(x), f[0].Diff(y), \
                f[1].Diff(x), f[1].Diff(y)), dims = (m,m))*P
        elif m == 3:
            gradient = P*CoefficientFunction((f[0].Diff(x), f[0].Diff(y), f[0].Diff(z), \
                f[1].Diff(x), f[1].Diff(y), f[1].Diff(z),\
                    f[2].Diff(x), f[2].Diff(y), f[2].Diff(z)), dims = (m,m))*P
            
    else:

        raise RuntimeError("Don't know how to take the gradient. Only scalars and vectors are accepted.")
    
    return gradient

Generate manufactured solution

In [3]:
# We include time to be more general possible
t = Parameter(0.0)

# generating mesh
fes_order = 2
mesh = generate_cylinder_mesh(maxh=0.1, order_g=fes_order+1)

# Definition of levelset for cilinder
R = 1.0
levelset_str = "y*y + z*z -" + str(R)
dimension = 3 # dimension of the embedding space

phi = CoefficientFunction(eval(levelset_str))

def compute_normal(phi, dim):
    if dim == 2:
        normal = CoefficientFunction((phi.Diff(x), phi.Diff(y)))
    elif dim ==3:
        normal = CoefficientFunction((phi.Diff(x), phi.Diff(y), phi.Diff(z)))
    else:
        raise RuntimeError("Don't know how to compute normal for dim not equal to 2 or 3")
    
    return normal/Norm(normal)

normal = compute_normal(phi,dimension)

P = Id(normal.dim) - OuterProduct(normal, normal)

# Exact velocity solution
_u = CoefficientFunction((1, 0, 0))
u_ex = (P*_u).Compile()

# Gradient
grad_u = gradient(u_ex,P).Compile()

# Divergence
div_u = Trace(grad_u).Compile()

# Symgrad
surface_strain = Sym(grad_u)
div_sym_grad_1 = Trace(gradient(surface_strain[0,:],P))
div_sym_grad_2 = Trace(gradient(surface_strain[1,:],P))
div_sym_grad_3 = Trace(gradient(surface_strain[2,:],P))

div_sym_grad = (P*CoefficientFunction((div_sym_grad_1, div_sym_grad_2, div_sym_grad_3))).Compile()

# Exact pressure solution
p_ex = CoefficientFunction(sin(x))

grad_p = gradient(p_ex,P)

f = -2.0*div_sym_grad + u_ex + grad_p
g = div_u


Defining the function solving the Stokes problem given the mesh, order of the finite element and the right-hand side

In [4]:
def steadyStokes(mesh, fes_order, f, g): 

    V = VectorH1(mesh, order=int(fes_order), dirichlet_bbnd="left|right")
    #################################
    ####### The command below should be Q = H1(mesh, order=int(fes_order-1)), i.e. without imposition of b.c.
    Q = H1(mesh, order=int(fes_order-1), dirichlet_bbnd="right|left")
    N = NumberSpace(mesh)
    W = V*Q*N

    (u, p, lam), (v, q, mu) = W.TnT()


    # Mesh size
    h = specialcf.mesh_size

    # Normal vector
    ns = specialcf.normal(mesh.dim)
    # normal and tangential  projection
    Qs = OuterProduct(ns, ns)
    Ps = Id(mesh.dim) - Qs
    # Weingarten map
    Bs = Grad(ns)

    def E_h(u):
        return Sym(Ps*Grad(u).Trace()*Ps)
    # return 0.5*Ps*(Grad(u).Trace())
    # return Ps*(grad(u).Trace())

    # Penalty parameter to weakly enforce tangential condition
    beta = 1.0

    # Corrected strain rate tensor for non-tangential 
    def E_th(u):
        return E_h(u) - InnerProduct(u, ns)*Bs

    Astar = BilinearForm(W)
    Astar += 2*InnerProduct(E_th(u), E_th(v))*ds
    # TODO: DO we need Traces here?
    Astar += InnerProduct(Ps*u.Trace(), Ps*v.Trace())*ds
    # TODO: Last term can replaced with
    # Astar += InnerProduct(u.Trace(), v.Trace())*ds
    # Penalization of normal component
    # TODO:More efficient way would be to compute (u,n)*(v*n) directly
    # Test that!
    Astar += beta*h**(-2.0)*InnerProduct(Qs*u.Trace(), Qs*v.Trace())*ds
    # Add b_h contributions 
    Astar += InnerProduct(grad(p).Trace(), v.Trace())*ds
    Astar += InnerProduct(grad(q).Trace(), u.Trace())*ds
    Astar += (p*mu + q*lam)*ds
    Astar.Assemble()


    l = LinearForm(W)
    l += ((Ps*f) * (Ps*v) - g*q )* ds 

    x_h = GridFunction(W)

    x_h.components[0].Set(u_ex, definedon=mesh.BBoundaries("right|left"))
    #################################
    ####### The command below should not be necessary
    x_h.components[1].Set(p_ex, definedon=mesh.BBoundaries("right|left"))

    res = l.vec.CreateVector()

    with TaskManager():
        Astar.Assemble()
        l.Assemble()

        res.data = l.vec
        res.data -= Astar.mat*x_h.vec

        x_h.vec.data += Astar.mat.Inverse(freedofs=W.FreeDofs()) * res.data

    return x_h

Running the simulation and computing the error

In [5]:
x_h = steadyStokes(mesh, fes_order, f, g)

u_h = x_h.components[0]
p_h = x_h.components[1]

# Printing .vtu file for visualization with Paraview
vtk = VTKOutput(mesh,coefs=[u_h, p_h, u_ex, p_ex],names=[ "u_h", "p_h",  "u", "p"],filename="solution")
vtk.Do(vb=BND)

err_u = InnerProduct(u_ex-u_h, u_ex-u_h)
err = sqrt(Integrate(cf = err_u, mesh=mesh, order = fes_order+2, VOL_or_BND = BND) )
print("error: ", err)

error:  3.5035994961462645e-05


In [6]:
def steadyStokesNoBoundary(mesh, fes_order, f, g): 

    V = VectorH1(mesh, order=int(fes_order), dirichlet_bbnd="left|right")
    #################################
    ####### The command below should be Q = H1(mesh, order=int(fes_order-1)), i.e. without imposition of b.c.
    Q = H1(mesh, order=int(fes_order-1), dirichlet_bbnd="right|left")
    N = NumberSpace(mesh)
    W = V*Q*N

    (u, p, lam), (v, q, mu) = W.TnT()


    # Mesh size
    h = specialcf.mesh_size

    # Normal vector
    ns = specialcf.normal(mesh.dim)
    # normal and tangential  projection
    Qs = OuterProduct(ns, ns)
    Ps = Id(mesh.dim) - Qs
    # Weingarten map
    Bs = Grad(ns)

    def E_h(u):
        return Sym(Ps*Grad(u).Trace()*Ps)
    # return 0.5*Ps*(Grad(u).Trace())
    # return Ps*(grad(u).Trace())

    # Penalty parameter to weakly enforce tangential condition
    beta = 1.0

    # Mass matrix scaling/friction
    Gamma=1.0

    # Viscosity
    eta = 1.0

    # Corrected strain rate tensor for non-tangential 
    def E_th(u):
        return E_h(u) - InnerProduct(u, ns)*Bs

    Astar = BilinearForm(W)
    Astar += 2*eta*InnerProduct(E_th(u), E_th(v))*ds
    # TODO: DO we need Traces here?
    Astar += Gamma*InnerProduct(Ps*u.Trace(), Ps*v.Trace())*ds
    # TODO: Last term can replaced with
    # Astar += InnerProduct(u.Trace(), v.Trace())*ds
    # Penalization of normal component
    # TODO:More efficient way would be to compute (u,n)*(v*n) directly
    # Test that!
    Astar += beta*h**(-2.0)*InnerProduct(Qs*u.Trace(), Qs*v.Trace())*ds
    # Add b_h contributions 
    Astar += InnerProduct(grad(p).Trace(), v.Trace())*ds
    Astar += InnerProduct(grad(q).Trace(), u.Trace())*ds
    Astar += (p*mu + q*lam)*ds
    Astar.Assemble()


    l = LinearForm(W)
    l += ((Ps*f) * (Ps*v) - g*q )* ds 

    x_h = GridFunction(W)

    #x_h.components[0].Set(u_ex, definedon=mesh.BBoundaries("right|left"))
    #################################
    ####### The command below should not be necessary
    #x_h.components[1].Set(p_ex, definedon=mesh.BBoundaries("right|left"))

    res = l.vec.CreateVector()

    with TaskManager():
        Astar.Assemble()
        l.Assemble()

        res.data = l.vec
        res.data -= Astar.mat*x_h.vec

        x_h.vec.data += Astar.mat.Inverse(freedofs=W.FreeDofs()) * res.data

    return x_h

In [38]:
torusMesh=generate_torus_mesh(0.1, order_g = 5)
Draw(torusMesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.23…

BaseWebGuiScene

In [39]:
# Normal vector
ns = specialcf.normal(torusMesh.dim)
# normal and tangential  projection
Qs = OuterProduct(ns, ns)
Ps = Id(mesh.dim) - Qs
# Weingarten map
Bs = Grad(ns)

In [45]:
VH=H1(torusMesh,order=4)
H_grid=GridFunction(VH)
H_grid.Set(H,definedon=torusMesh.Boundaries(".*"))
Draw(H_grid)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.23…

BaseWebGuiScene

In [52]:
gradH=Cross(Ps*Grad(H_grid),ns)
Draw(gradH,torusMesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.23…

BaseWebGuiScene

In [53]:
vtk = VTKOutput(torusMesh,coefs=[H,gradH],names=[ "H", "gradH"],filename="curvatureTest",subdivision=3)
vtk.Do(vb=BND)

'curvatureTest'